# 72 — Build LGBM features + train LambdaRank (Stage C)

Walks HF train conversations, splits sessions 80/20, runs the full
Stage A+B retrieval+reranker pipeline to get top-100 candidates per
music turn, then extracts the extended 28-feature vectors per
(turn, candidate) pair. Trains LightGBM LambdaRank on the result.

**Prereqs**: Stage A + Stage B done; merged BGE-M3 + CE on Hub;
BGE-M3-FT catalog pickle on Drive (notebook 70 cell 6).

**Wallclock**: ~4-6 hr on Blackwell (feature extraction is wRRF +
CE forward over ~12k music turns × 100 cands).

In [ ]:
# 1) Setup. Disable JAX GPU preallocation BEFORE any import pulls JAX in.
# datasets/transformers import JAX transitively; JAX grabs ~75% of VRAM on
# first use, so the KERNEL ends up hogging the GPU and the cell-3 !python
# subprocess OOMs. This is why nb 72 OOM'd while nb 70/71 (which set these)
# did not. If the kernel already imported JAX, RESTART RUNTIME for this to take.
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH', 'true')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

BRANCH = 'recall-union-lgbm'  # G2: 3-channel union pool + new session features + album_name fix
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
src = f'{DRIVE_BASE}/recsys2026_retrieval_v2_cache'
dst = f'{LOCAL_BASE}/retrieval_v2'
if os.path.islink(dst): os.unlink(dst)
elif os.path.exists(dst):
    import shutil; shutil.rmtree(dst)
os.symlink(src, dst)

# Retrieval stack (bm25->bm25s, dense->sentence-transformers/peft) is imported
# eagerly by mcrs.retrieval_modules, so its deps are required even for the
# LGBM build. Matches nb 71's proven set + lightgbm/scikit-learn.
!pip install -q --upgrade 'transformers>=4.40' 'accelerate>=0.30' 'peft>=0.11' \
    'datasets' 'pandas<3.0' 'tqdm' 'huggingface_hub' 'sentence-transformers>=3.0' \
    'FlagEmbedding>=1.3' 'bm25s' 'lightgbm' 'scikit-learn'

In [ ]:
# 2) Walk HF train conversations + session-disjoint 80/20 split.
import sys
sys.path.insert(0, '/content/recsys2026/scripts')
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from build_bi_encoder_training_data import _iter_conversation_turns

train_conv = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='train')
all_rows = _iter_conversation_turns(train_conv)
print(f'{len(all_rows)} per-music-turn rows from train split')
session_ids = sorted({r['session_id'] for r in all_rows})
train_sids, val_sids = train_test_split(session_ids, test_size=0.2, random_state=42)
train_set, val_set = set(train_sids), set(val_sids)
train_rows = [r for r in all_rows if r['session_id'] in train_set]
val_rows = [r for r in all_rows if r['session_id'] in val_set]
print(f'train turns: {len(train_rows)}  val turns: {len(val_rows)}')
import json as _j
os.makedirs('experiments/cache/retrieval_v2/lgbm', exist_ok=True)
with open('experiments/cache/retrieval_v2/lgbm/lgbm_train_rows.jsonl', 'w') as f:
    for r in train_rows: f.write(_j.dumps(r, default=str) + '\n')
with open('experiments/cache/retrieval_v2/lgbm/lgbm_val_rows.jsonl', 'w') as f:
    for r in val_rows: f.write(_j.dumps(r, default=str) + '\n')

In [ ]:
# 3) Extract features for each (turn, candidate) pair via Stage A+B pipeline.
# For each music turn:
#   a. Build production query via format_query_text(..., mode='bge_m3_structured').
#   b. wrrf_union_v1 (BM25 + dense_metadata_qwen3 + same_artist) → top-100 candidates.
#   c. extract_features() builds the per-candidate feature row (no cross-encoder).
# Streams live to the cell AND tees a log to Drive; watch the tqdm 'wrrf batches'
# bar for ETA and the final '[lgbm-features] timing: union=... features=...' line.
# Outputs: experiments/cache/retrieval_v2/lgbm/lgbm_{train,val}_features.parquet
# n-sessions=5000 = fast directional run; raise to 999999 for the FINAL model.
!cd /content/recsys2026/music-crs-baselines && python -u ../scripts/build_lgbm_features.py \
    --n-sessions 5000 \
    --topk 100 \
    --seed 42 \
    --out /content/recsys2026/experiments/cache/retrieval_v2/lgbm/lgbm_train_features.parquet \
    --cache-dir /content/recsys2026/experiments/cache \
    2>&1 | tee /content/drive/MyDrive/recsys2026_retrieval_v2_cache/lgbm_build_log.txt
# Note: the existing build_lgbm_features.py samples train sessions; with the
# new 80/20 split, override the sampler by writing a thin per-row driver.
# Implementation detail: pass session_ids filter via the existing --seed +
# n-sessions, OR modify build_lgbm_features.py to accept --session-id-list.
# For Phase 1, the simpler path is: run the full feature extractor on train,
# then post-filter rows to (train_sids, val_sids) into two parquets.

In [ ]:
# 4) Post-filter the single full-train parquet into 80/20 train/val by session.
import pandas as pd
df = pd.read_parquet('/content/recsys2026/experiments/cache/retrieval_v2/lgbm/lgbm_train_features.parquet')
tdf = df[df['session_id'].isin(train_set)].copy()
vdf = df[df['session_id'].isin(val_set)].copy()
tdf.to_parquet('/content/recsys2026/experiments/cache/retrieval_v2/lgbm/lgbm_train_split.parquet', index=False)
vdf.to_parquet('/content/recsys2026/experiments/cache/retrieval_v2/lgbm/lgbm_val_split.parquet', index=False)
print(f'train rows: {len(tdf)}  val rows: {len(vdf)}')
print('positives (label=1):', int(tdf['label'].sum()), int(vdf['label'].sum()))

In [ ]:
# 5) Train LightGBM LambdaRank.
!cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
    --train-features experiments/cache/retrieval_v2/lgbm/lgbm_train_split.parquet \
    --val-features   experiments/cache/retrieval_v2/lgbm/lgbm_val_split.parquet \
    --output-dir     experiments/cache/retrieval_v2/lgbm/lgbm_v1 \
    --n-estimators 1000 \
    2>&1 | tee /content/drive/MyDrive/recsys2026_retrieval_v2_cache/lgbm_train_log.txt
!ls -la /content/recsys2026/experiments/cache/retrieval_v2/lgbm/lgbm_v1/

In [ ]:
# 6) Ensure the trained model is on Drive for inference reuse.
# NOTE: experiments/cache/retrieval_v2 is symlinked to the Drive cache (cell 2),
# so cell 5 already wrote lgbm_v1 ONTO Drive. src and dst below resolve to the
# SAME directory; the old rmtree(dst)+copytree(src) therefore DELETED the model
# and then failed. Guard against that: skip the copy when they're the same path.
import os, shutil
src = '/content/recsys2026/experiments/cache/retrieval_v2/lgbm/lgbm_v1'
dst = '/content/drive/MyDrive/recsys2026_retrieval_v2_cache/lgbm/lgbm_v1'
if not os.path.exists(src):
    raise FileNotFoundError(f'{src} missing — run cell 5 (training) first.')
if os.path.realpath(src) == os.path.realpath(dst):
    print('Model already on Drive via the retrieval_v2 symlink — nothing to copy:')
    print(' ', os.path.realpath(dst))
    print('  contents:', sorted(os.listdir(src)))
else:
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print('LGBM model dir mirrored to:', dst)


In [ ]:
# 7) Dev eval (FULL dev): deep union pool -> LGBM rerank. NO cross-encoder.
# Tests the recall-union DEPTH hypothesis. The live pipeline RRF-fuses each
# source's top-100 then truncates the FUSED list to 100, which discards the
# union pool's recall (audit pool ceiling ~0.49) down to recall@100~=0.43. Here
# we retrieve a DEEP pool (POOL_DEPTH) and report recall@{20,100,POOL_DEPTH}
# ceilings plus nDCG@20 for union-only vs LGBM@100 vs LGBM@deep, over the FULL
# dev split (not the old first-1000-turns sample, which was only ~125 sessions
# and underpowered the lift CI).
#
# Queries are built EXACTLY like build_lgbm_features.py (raw 'role: content'
# lines, music turns expanded via id_to_metadata) so the text matches training.
#
# TRAIN/SERVE CAVEAT: the LGBM was trained on depth-100 pools, so its `wrrf_rank`
# feature (candidate position in the input list, lgbm_rerank.py:310) is
# in-distribution only for ranks 1..100. At POOL_DEPTH=300 the tail (101..300)
# is extrapolated, so LGBM@deep is a LOWER bound on what a depth-matched model
# could do. The recall ceilings are model-free and unaffected. Decision rule:
#   - recall@POOL_DEPTH vs the 0.46 G1 gate answers "is the ceiling there?"
#   - if LGBM@deep already beats LGBM@100, the deeper pool helps even OOD -> ship
#     it (config 190 / nb73) AND retrain the feature builder (cell 3) at the
#     matching depth to remove the wrrf_rank skew.
import sys, math
import numpy as np
import pandas as pd
from datasets import load_dataset
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
from mcrs.db_item.music_catalog import MusicCatalogDB
from mcrs.retrieval_modules import load_retrieval_module
from mcrs.rerankers import load_reranker_module

ITEM_DB    = 'talkpl-ai/TalkPlayData-Challenge-Track-Metadata'
CORPUS     = ['track_name', 'artist_name', 'album_name']
CACHE_DIR  = '/content/recsys2026/experiments/cache'
LGBM_DIR   = '/content/recsys2026/experiments/cache/retrieval_v2/lgbm/lgbm_v1'
POOL_DEPTH = 300     # deep union pool fed to the reranker (was 100)
N_EVAL     = None    # None = FULL dev split (all music turns); set an int to cap

item_db = MusicCatalogDB(ITEM_DB, ['all_tracks'], CORPUS)

# Build dev queries the SAME way as the training feature builder (raw mode).
dev = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')
queries, golds, user_ids, played, goal_cats, goal_specs, turn_nums = [], [], [], [], [], [], []
for sess in dev:
    if N_EVAL is not None and len(queries) >= N_EVAL:
        break
    df = pd.DataFrame(sess['conversations'])
    cg = sess.get('conversation_goal') or {}
    for _, music in df[df['role'] == 'music'].iterrows():
        if N_EVAL is not None and len(queries) >= N_EVAL:
            break
        turn_n = int(music['turn_number'])
        prior = df[(df['turn_number'] < turn_n) |
                   ((df['turn_number'] == turn_n) & (df['role'] == 'user'))]
        lines = []
        for _, t in prior.iterrows():
            role = 'assistant' if t['role'] == 'music' else t['role']
            content = t['content']
            if t['role'] == 'music':
                try:
                    content = item_db.id_to_metadata(content)
                except Exception:
                    content = str(content)
            lines.append(f'{role}: {content}')
        queries.append(chr(10).join(lines))
        golds.append(music['content'])
        user_ids.append(sess.get('user_id'))
        played.append(list(df[(df['role'] == 'music') & (df['turn_number'] < turn_n)]['content']))
        goal_cats.append(cg.get('category'))
        goal_specs.append(cg.get('specificity'))
        turn_nums.append(turn_n)
print('[dev eval] built', len(queries), 'dev queries', '(FULL split)' if N_EVAL is None else '(capped)')

# Stage A: one DEEP union retrieval. same_artist channel reads ctx['history_tids'].
# poolD is RRF-ordered, so poolD[:100] reproduces the old fused-100 exactly.
union = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR, extra_config={})
poolD = union.batch_text_to_item_retrieval(
    queries, topk=POOL_DEPTH, user_ids=user_ids,
    batch_context=[{'history_tids': p} for p in played])

# Stage C: LGBM rerank top-20. session features read ctx['played_tids'].
lgbm = load_reranker_module('lgbm_rerank', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR, model_path=LGBM_DIR)
def _rerank(cands):
    return lgbm.rerank(
        queries, cands, topk=20, user_ids=user_ids,
        goal_categories=goal_cats, goal_specificities=goal_specs,
        extra_session_info=[{'played_tids': p, 'turn_number': tn}
                            for p, tn in zip(played, turn_nums)])
rr_100  = _rerank([c[:100] for c in poolD])  # baseline: rerank the fused top-100
rr_deep = _rerank(poolD)                     # treatment: rerank the deep pool

def ndcg20(ranked, gold):
    r = ranked[:20]
    return 1.0 / math.log2(r.index(gold) + 2) if gold in r else 0.0
def recall_at(cands, k):
    return float(np.mean([1.0 if g in c[:k] else 0.0 for c, g in zip(cands, golds)]))

union20  = np.array([ndcg20(c, g) for c, g in zip(poolD, golds)])    # union order, top-20
ndcg_100 = np.array([ndcg20(r, g) for r, g in zip(rr_100, golds)])   # LGBM @ depth-100
ndcg_dp  = np.array([ndcg20(r, g) for r, g in zip(rr_deep, golds)])  # LGBM @ deep pool

def boot_ci(diff):
    rng = np.random.default_rng(0)
    b = diff[rng.integers(0, len(diff), size=(2000, len(diff)))].mean(axis=1)
    return float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))

print('=== Dev nDCG@20 (n=' + str(len(golds)) + ', FULL dev) ===')
print('  recall ceilings   : @20=' + str(round(recall_at(poolD, 20), 4)) +
      '  @100=' + str(round(recall_at(poolD, 100), 4)) +
      '  @' + str(POOL_DEPTH) + '=' + str(round(recall_at(poolD, POOL_DEPTH), 4)) +
      '   (G1 gate 0.46)')
print('  union only  (top20):', round(float(union20.mean()), 4))
print('  union + LGBM @100  :', round(float(ndcg_100.mean()), 4))
print('  union + LGBM @' + str(POOL_DEPTH) + '  :', round(float(ndcg_dp.mean()), 4))
for label, base, new in [
        ('LGBM@100   - union   ', union20,  ndcg_100),
        ('LGBM@' + str(POOL_DEPTH) + '  - union   ', union20,  ndcg_dp),
        ('LGBM@' + str(POOL_DEPTH) + '  - LGBM@100', ndcg_100, ndcg_dp)]:
    d = new - base
    lo, hi = boot_ci(d)
    print('  ' + label + ' :', round(float(d.mean()), 4),
          '95% CI [', round(lo, 4), ',', round(hi, 4), ']',
          'SIGNIFICANT' if lo > 0 else 'not significant')
print('  (Blind-A baseline 0.09 is END-TO-END; these are retrieval-stage -> directional only)')


In [ ]:
# 8) Phase-1 recall ablation: union without vs with the HyDE channel.
# Reuses queries/golds/played/user_ids built in cell 7. First run cell 7 with
# N_EVAL=1500 to smoke-test the LLM + channel wiring (HyDE cache populates under
# CACHE_DIR/hyde/), then N_EVAL=None for the full-dev number. Decision gate:
# if union+HyDE recall@100 >= baseline + 0.03, proceed to Phase 2 (reranker
# features); if flat, swap a stronger embedder for the HyDE channel first.
#
# HyDE generation is BATCHED (left-padded, longest-first) and cache-miss-only,
# so the full-dev pass is ~1-2h on an L4 (vs many hours unbatched); re-runs hit
# the cache and are instant. If you have Qwen2.5-7B cached on Drive, point
# hyde_model at it to skip the ~15GB download. Find it with:
#     !ls /content/drive/MyDrive | grep -i qwen
import numpy as np
from mcrs.retrieval_modules import load_retrieval_module

# HyDE channel knobs (explicit; values equal the factory defaults). The
# generation model is Qwen2.5-7B-Instruct — loaded via the generic HF wrapper
# (LLAMA_MODEL), NOT Llama. Tune here without touching the library.
HYDE_CFG = {
    'use_hyde': True,
    'w_hyde': 1.0,                             # HyDE channel weight in the wRRF fusion
    'hyde_model': 'Qwen/Qwen2.5-7B-Instruct',  # HF repo id OR a local Drive path
    'hyde_n_docs': 3,                          # pseudo-tracks generated per turn
    'hyde_topk_per_doc': 100,                  # dense candidates per pseudo-track before RRF
    'hyde_batch_size': 16,                     # conversations per generation forward pass
}

def recall_at(cands, k):
    return float(np.mean([1.0 if g in c[:k] else 0.0 for c, g in zip(cands, golds)]))

base = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS,
                             CACHE_DIR, extra_config={})
hyde = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS,
                             CACHE_DIR, extra_config=HYDE_CFG)

ctx = [{'history_tids': p} for p in played]
cb = base.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
ch = hyde.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)

print('=== Phase-1 recall ablation (n=' + str(len(golds)) + ', FULL dev) ===')
print('  union (3-chan)   : recall@20=' + str(round(recall_at(cb, 20), 4)) +
      ' @100=' + str(round(recall_at(cb, 100), 4)))
print('  union + HyDE     : recall@20=' + str(round(recall_at(ch, 20), 4)) +
      ' @100=' + str(round(recall_at(ch, 100), 4)) + '   (G1 gate 0.46)')
print('  delta recall@100 :', round(recall_at(ch, 100) - recall_at(cb, 100), 4))


In [ ]:
# 10) SASRec channel recall ablation (Colab). Prerequisites (run first):
#   a. Confirm the CLAP audio column name:
#        from datasets import load_dataset
#        ds = load_dataset('talkpl-ai/TalkPlayData-Challenge-Track-Embeddings', split='all_tracks')
#        print([c for c in ds.column_names if 'clap' in c.lower() or 'audio' in c.lower()])
#      If it is NOT 'laion_clap', set CLAP_COL in scripts/train_sasrec.py and re-commit.
#   b. Train (Colab GPU, train split only):
#        !cd /content/recsys2026 && python -u scripts/train_sasrec.py \
#            --cache-dir /content/recsys2026/experiments/cache --out sasrec_v1 --epochs 5
# This cell builds dev data (incl. user-turns dialog), checks dialog lengths vs
# bge-base-en's 512 cap, then compares union recall@100 WITHOUT vs WITH the
# SASRec channel. Decision gate: if union+SASRec recall@100 lifts meaningfully
# (target >= +0.03, clears 0.46), proceed to P1 (the LGBM relevance feature).
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer
from mcrs.db_item.music_catalog import MusicCatalogDB
from mcrs.retrieval_modules import load_retrieval_module
from mcrs.retrieval_modules.sasrec_model import build_user_dialog

# Shared constants (self-contained — no dependency on earlier cells).
ITEM_DB = 'talkpl-ai/TalkPlayData-Challenge-Track-Metadata'
CORPUS = ['track_name', 'artist_name', 'album_name']
CACHE_DIR = '/content/recsys2026/experiments/cache'

item_db = MusicCatalogDB(ITEM_DB, ['all_tracks'], CORPUS)
dev = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')
queries, golds, user_ids, played, user_dialogs = [], [], [], [], []
N_SASREC_EVAL = None  # None = full dev; set an int to cap for a smoke run
for sess in dev:
    if N_SASREC_EVAL is not None and len(queries) >= N_SASREC_EVAL:
        break
    df = pd.DataFrame(sess['conversations'])
    for _, music in df[df['role'] == 'music'].iterrows():
        if N_SASREC_EVAL is not None and len(queries) >= N_SASREC_EVAL:
            break
        tn = int(music['turn_number'])
        prior = df[(df['turn_number'] < tn) |
                   ((df['turn_number'] == tn) & (df['role'] == 'user'))]
        lines = []
        for _, t in prior.iterrows():
            role = 'assistant' if t['role'] == 'music' else t['role']
            content = item_db.id_to_metadata(t['content']) if t['role'] == 'music' else t['content']
            lines.append(f'{role}: {content}')
        queries.append(chr(10).join(lines))
        user_dialogs.append(build_user_dialog(prior.to_dict('records')))
        golds.append(music['content'])
        user_ids.append(sess.get('user_id'))
        played.append(list(df[(df['role'] == 'music') & (df['turn_number'] < tn)]['content']))
print('[sasrec eval] built', len(queries), 'dev turns')

# Dialog-length check: how many user-dialogs exceed bge-base-en's 512-token cap?
tok = AutoTokenizer.from_pretrained('BAAI/bge-base-en-v1.5')
lens = [len(tok.encode(d)) for d in user_dialogs]
print('[sasrec eval] user-dialog tokens: median', int(np.median(lens)),
      ' p95', int(np.percentile(lens, 95)),
      ' frac>512:', round(float(np.mean([l > 512 for l in lens])), 4))

def recall_at(cands, k):
    return float(np.mean([1.0 if g in c[:k] else 0.0 for c, g in zip(cands, golds)]))

base = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS,
                             CACHE_DIR, extra_config={})
sas = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS,
                            CACHE_DIR, extra_config={'use_sasrec': True, 'w_sasrec': 1.0})
ctx = [{'history_tids': p, 'user_dialog': ud} for p, ud in zip(played, user_dialogs)]
cb = base.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
cs = sas.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
print('=== SASRec channel recall ablation (n=' + str(len(golds)) + ', FULL dev) ===')
print('  union (3-chan) : recall@20=' + str(round(recall_at(cb, 20), 4)) +
      ' @100=' + str(round(recall_at(cb, 100), 4)))
print('  union + SASRec : recall@20=' + str(round(recall_at(cs, 20), 4)) +
      ' @100=' + str(round(recall_at(cs, 100), 4)) + '   (G1 gate 0.46)')
print('  delta recall@100 :', round(recall_at(cs, 100) - recall_at(cb, 100), 4))
